In [40]:
import pyspark

In [41]:
from pyspark.sql import SparkSession

In [42]:
spark_job = SparkSession.builder.appName("Data Analysis Job").getOrCreate()
spark_job

### 1. Data Inspection

We start by loading the raw CSV and checking the schema and sample rows. The first goal is to understand how the data is structured before cleaning or coercing columns.

In [43]:
raw_df = (
    spark_job.read.option("header", "true")
    .option("sep", ",")
    .option("quote", '"')
    .option("escape", '"')
    .option("inferSchema", "false")
    .csv("salaries.csv")
)
raw_df.show(10, truncate=False)

+-----------------+----------------------------------------------------------+------------------------------+------------------------------------------------------------+---------+------------------------------------------------------------+-------------------------------------------------------------------+----------+-------------+--------------------+-----------------+
|entidadfederativa|sujetoobligado                                            |nombre                        |denominacion                                                |montoneto|cargo                                                       |area                                                               |montobruto|idInformacion|periodoreportainicio|periodoreportafin|
+-----------------+----------------------------------------------------------+------------------------------+------------------------------------------------------------+---------+------------------------------------------------------------+-----------

### 2. Data Cleaning and Missing Values

We inspect missing values and identify whether nulls are concentrated in critical columns. Since `montobruto` and `montoneto` are the central business fields, we treat them as the primary quality checks.

In [5]:
# Null Count
null_count = raw_df.count() - raw_df.na.drop().count()
null_count

187975

In [6]:
# checking for null percent compared to original dataset count
null_count_pct = (null_count/raw_df.count())*100
null_count_pct

9.350682814989545

### 3. Salary Validation: Gross and Net Pay

The salary columns contained inconsistent formatting. Some values included commas, blanks, aliases like `NULL`, or text that should not be numeric. We validated them before converting to doubles.

In [7]:
raw_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: string (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: string (nullable = true)
 |-- periodoreportafin: string (nullable = true)



In [8]:
# only care about rows having actual salary figures which is the core of this dataset
raw_df.na.drop(subset=['montobruto', 'montoneto']).count()

1861815

In [9]:
pct_with_salaries = (raw_df.na.drop(subset=['montobruto', 'montoneto']).count())/(raw_df.count())*100
pct_with_salaries

92.61466431807295

Showing top states with largest null count

In [10]:
dropped_rows = raw_df.subtract(raw_df.na.drop(subset=['montobruto', 'montoneto']))
dropped_rows.groupBy('entidadfederativa').count().orderBy('count', ascending=False).show()

+--------------------+-----+
|   entidadfederativa|count|
+--------------------+-----+
|             Jalisco|27641|
|     Baja California|25384|
|            Guerrero|15813|
|     San Luis Potosí| 6006|
|              Sonora| 4383|
|          Federación| 3071|
|      Aguascalientes| 2316|
|              Colima| 1070|
|             Hidalgo|  934|
|             Chiapas|  574|
|Coahuila de Zaragoza|  539|
|    Ciudad de México|  223|
|           Chihuahua|  177|
| Michoacán de Ocampo|  124|
|          Guanajuato|  115|
|             Morelos|   84|
|             Tabasco|   49|
| Baja California Sur|   37|
|          Nuevo León|   36|
|           Querétaro|   24|
+--------------------+-----+
only showing top 20 rows


### DATA VALIDATION

In [11]:
raw_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: string (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: string (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: string (nullable = true)
 |-- periodoreportafin: string (nullable = true)



We need to convert  montobruto and montoneto to double and both periods to datetime

In [15]:
spark_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)



### 4. Data Quality Issue and Fix

The first attempts failed because the dataset had inconsistent formats in the raw CSV. Converting the fields too early caused parsing errors. The fix was to read the raw data as text, validate values with regex, clean the fields, and only then cast to numeric/date types.

Exact troubleshooting step:

We inspected the raw CSV as strings and filtered for values that were clearly not valid numeric salary entries. This is how we could see values such as `USET`, `NULL`, `NA`, empty strings, and date-like text appearing inside `montobruto` and `montoneto` before any cast to double.



This made the anomaly visible before conversion and showed that the salary columns were not clean numeric fields.

In [16]:
from pyspark.sql import functions as F
# Find bad rows before casting
bad_salary_rows = raw_df.filter(
    F.trim(F.col("montobruto").cast("string")).rlike(r".*[A-Za-z].*")
    |
    F.trim(F.col("montoneto").cast("string")).rlike(r".*[A-Za-z].*")
    |
    F.trim(F.col("montobruto").cast("string")).isin("USET", "NULL", "NA", "")
    |
    F.trim(F.col("montoneto").cast("string")).isin("USET", "NULL", "NA", "")
    |
    F.trim(F.col("montobruto").cast("string")).rlike(r".*\d{2}/\d{2}/\d{4}.*")
    |
    F.trim(F.col("montoneto").cast("string")).rlike(r".*\d{2}/\d{2}/\d{4}.*")
    # |
    # ~F.trim(F.col("periodoreportainicio").cast("string")).rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$")
    # |
    # ~F.trim(F.col("periodoreportafin").cast("string")).rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$")
)

bad_salary_rows.select(
    "entidadfederativa",
    "montobruto",
    "montoneto"
    # "periodoreportainicio",
    # "periodoreportafin"
).show(50, truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------------------------------+-----------------+
|entidadfederativa                                                                                                                                                                                                                                                                      |montobruto                                  |montoneto        |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----------------------------

In [44]:
# Validate raw strings before converting them to analysis types.
clean_df = (
    raw_df
    .withColumn(
        "montobruto",
        F.when(
            F.trim(F.col("montobruto")).rlike(r"^[0-9,.\s-]+$"),
            F.regexp_replace(F.trim(F.col("montobruto")), ",", "").cast("double")
        )
    )
    .withColumn(
        "montoneto",
        F.when(
            F.trim(F.col("montoneto")).rlike(r"^[0-9,.\s-]+$"),
            F.regexp_replace(F.trim(F.col("montoneto")), ",", "").cast("double")
        )
    )
    .withColumn(
        "periodoreportainicio",
        F.trim(F.col("periodoreportainicio"))
    )
    .withColumn(
        "periodoreportafin",
        F.trim(F.col("periodoreportafin"))
    )
    .filter(
        F.col("montobruto").isNotNull()
        & F.col("montoneto").isNotNull()
        & F.col("periodoreportainicio").rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$")
        & F.col("periodoreportafin").rlike(r"^\d{2}/\d{2}/\d{4}$|^\d{4}-\d{2}-\d{2}$")
    )
)

clean_df.select(
    "montobruto", "montoneto", "periodoreportainicio", "periodoreportafin"
).show(20, truncate=False)

+----------+---------+--------------------+-----------------+
|montobruto|montoneto|periodoreportainicio|periodoreportafin|
+----------+---------+--------------------+-----------------+
|4254.0    |4000.0   |01/01/2018          |30/06/2018       |
|16092.0   |12177.86 |01/01/2018          |31/03/2018       |
|16030.0   |11652.0  |01/01/2020          |31/03/2020       |
|2910.65   |10180.57 |01/07/2018          |31/12/2018       |
|6188.4    |17004.4  |01/07/2019          |31/12/2019       |
|2982.2    |2644.0   |01/07/2018          |31/12/2018       |
|4898.28   |4898.28  |01/07/2019          |31/12/2019       |
|619.0     |474.96   |01/05/2019          |31/05/2019       |
|16812.67  |13109.02 |01/04/2018          |30/06/2018       |
|7579.22   |6299.3   |01/04/2018          |30/06/2018       |
|3974.8    |13730.55 |01/07/2018          |31/12/2018       |
|13280.15  |11753.87 |01/04/2019          |30/06/2019       |
|13198.8   |12000.0  |01/08/2019          |31/08/2019       |
|3918.15

### 5. Feature Engineering

After cleaning the required fields, we compute:
- `period_dias`: number of days between report start and end
- `deducciones_estimadas`: difference between gross and net salary

In [45]:
# Parse both date formats only after the raw values pass validation.
clean_df = clean_df.withColumn(
    "periodoreportainicio",
    F.coalesce(
        F.to_date("periodoreportainicio", "dd/MM/yyyy"),
        F.to_date("periodoreportainicio", "yyyy-MM-dd")
    )
).withColumn(
    "periodoreportafin",
    F.coalesce(
        F.to_date("periodoreportafin", "dd/MM/yyyy"),
        F.to_date("periodoreportafin", "yyyy-MM-dd")
    )
)

clean_df = clean_df.filter(
    F.col("periodoreportainicio").isNotNull()
    & F.col("periodoreportafin").isNotNull()
)

clean_df = clean_df.withColumn(
    "period_dias",
    F.datediff("periodoreportafin", "periodoreportainicio")
).withColumn(
    "deducciones_estimadas",
    F.col("montobruto") - F.col("montoneto")
)

clean_df.printSchema()
clean_df.select(
    "montobruto", "montoneto",
    "periodoreportainicio", "periodoreportafin",
    "period_dias", "deducciones_estimadas"
).show(20, truncate=False)

clean_df.select(
    "montobruto", "montoneto", "period_dias", "deducciones_estimadas"
).describe().show()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)
 |-- deducciones_estimadas: double (nullable = true)

+----------+---------+--------------------+-----------------+-----------+---------------------+
|montobruto|montoneto|periodoreportainicio|periodoreportafin|period_dias|deducciones_estimadas|
+----------+---------+--------------------+-----------------+-----------+---------------------+
|4254.0    |4000.0   |2018-01-01          |2018-06-30       |180        |254.0                |
|16092.0   |12177.86 |2018-01-01      

### EXPLORATORY DATA ANALYSIS

In [19]:
from pyspark.sql import functions as F

clean_df.printSchema()
clean_df.columns
clean_df.select("montobruto", "montoneto").show(20, truncate=False)

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)
 |-- deducciones_estimadas: double (nullable = true)

+----------+---------+
|montobruto|montoneto|
+----------+---------+
|4254.0    |4000.0   |
|16092.0   |12177.86 |
|16030.0   |11652.0  |
|2910.65   |10180.57 |
|6188.4    |17004.4  |
|2982.2    |2644.0   |
|4898.28   |4898.28  |
|619.0     |474.96   |
|16812.67  |13109.02 |
|7579.22   |6299.3   |
|3974.8    |13730.55 |
|13280.15  |11753.87 |
|13198.8   |12000.0  |
|3918.15   |3673.91  |
|14861.81  |12050.8  |
|27920.1

Highest earners — select + orderBy

In [20]:
clean_df.select('nombre', 'cargo', 'montobruto').orderBy('montobruto', ascending=False).show(10)

+--------------------+--------------------+----------+
|              nombre|               cargo|montobruto|
+--------------------+--------------------+----------+
|MYRIAM VILMA URZU...|Secretaria de Pro...| 7559543.0|
|CARLOS MIGUEL SAI...|Subsecretario de ...| 7290065.0|
|HUMBERTO GONZALEZ...|Director de Vincu...| 7008311.0|
|Francisco David V...|              Asesor| 5101137.0|
|JUAN GERARDO VARG...|Coordinadora de P...| 3257057.0|
|Ramon Leonardo Go...|                NULL|2237386.03|
|Ramon Leonardo Go...|                NULL|2237386.03|
|Francisco Martíne...|Líder Coordinador...| 1699079.0|
|                NULL|     Jefe De Oficina|1662886.88|
|Emanuel Ortiz Rod...|Secretaria de dep...| 1497404.0|
+--------------------+--------------------+----------+
only showing top 10 rows


Data quality check for any remaining zero/negative salaries

In [21]:
clean_df.filter((clean_df['montobruto'] <= 0) | (clean_df['montoneto'] <= 0)).count()

54182

Average gross salary by state

In [22]:
clean_df.groupBy('entidadfederativa').agg(
    F.count('*').alias('record_count'),
    F.avg('montobruto').alias('avg_gross')
).orderBy('avg_gross', ascending=False).show()

+--------------------+------------+------------------+
|   entidadfederativa|record_count|         avg_gross|
+--------------------+------------+------------------+
|             Sinaloa|        1003| 53523.21579262213|
| Baja California Sur|       23676|40842.159535394494|
|           Chihuahua|       64905|35127.899206686714|
|",LAURA CARINO SA...|           1|          34734.44|
|             Nayarit|        3225| 23797.99314418605|
| Michoacán de Ocampo|        9399|22355.339324396205|
|     Baja California|       53943|21806.607663830437|
|          Federación|      673652|19383.365629107986|
|             Tabasco|        9447| 19336.66011538053|
|              Sonora|        6841|17977.468738488533|
|",JUAN CARLOS LOP...|           1|          17213.16|
|Coahuila de Zaragoza|       41444| 16890.59274683912|
|              México|        2222|16820.915018001793|
|",SONIA OLIVAS AV...|           1|           16361.5|
|              Colima|       48628| 15843.31278748872|
|         

Agencies with the most records

In [23]:
clean_df.groupBy('sujetoobligado').agg(F.count('*').alias('record_count')).orderBy('record_count', ascending=False).show(10)

+--------------------+------------+
|      sujetoobligado|record_count|
+--------------------+------------+
|Instituto Mexican...|      126635|
|INSTITUTO DE EDUC...|      104550|
|Autoridad Educati...|       96230|
|Secretaría de Edu...|       94685|
|Secretaría de Edu...|       67436|
|Secretaría de Bie...|       48307|
|Comisión Federal ...|       40284|
|Instituto Politéc...|       39218|
|    Policía Auxiliar|       34492|
|Secretaría de Seg...|       32986|
+--------------------+------------+
only showing top 10 rows


Deduction rate — withColumn

In [24]:
clean_df = clean_df.withColumn(
    'deduction_pct',
    ((clean_df['montobruto'] - clean_df['montoneto']) / clean_df['montobruto']) * 100
)
clean_df.select('nombre', 'cargo', 'deduction_pct').filter(clean_df['montobruto'] > 0).orderBy('deduction_pct', ascending=False).show(10)

+--------------------+--------------------+------------------+
|              nombre|               cargo|     deduction_pct|
+--------------------+--------------------+------------------+
|  JOSE MORALES GOMEZ|LIDER COORDINADOR...| 4698.443815683105|
|JOSE HERNANDEZ SORIA|LIDER COORDINADOR...|3024.4868797090153|
|MARIA ANGELICA PE...| ENCARGADO DE TALLER|             656.0|
|JAZMIN ALEJANDRA ...|ENCARGADO DE EVAL...| 599.5272727272727|
|SARA MERCEDES PRI...|PROF. INVEST. ASO...|377.66385939216406|
|JOSE DE JESUS SAN...|             DOCENTE|340.02777777777777|
|ANDREA SANCHEZ MA...|SECRETARIA DE REC...|260.42865362485617|
|MIRIAM CECILIA RA...|SECRETARIA DE CON...|237.62558369888214|
|JOSE VALENTIN FLO...|ENCARGADO DE PROY...| 206.4022933588151|
|PEDRO MARTINEZ MORAN|          CAJERO (A)| 197.9585629645851|
+--------------------+--------------------+------------------+
only showing top 10 rows


Average pay by position (cargo)

In [25]:
clean_df.groupBy('cargo').agg(
    F.avg('montobruto').alias('avg_gross'),
    F.count('*').alias('count')
).orderBy('avg_gross', ascending=False).show(20)

+--------------------+-----------------+-----+
|               cargo|        avg_gross|count|
+--------------------+-----------------+-----+
|Secretaria de Pro...|        7559543.0|    1|
|Subsecretario de ...|        7290065.0|    1|
|Director de Vincu...|        3504155.5|    2|
|Líder Coordinador...|        1699079.0|    1|
|Coordinadora de P...|        1648728.5|    2|
|   Asistecial Medica|        1315650.0|    1|
|Administrativo Es...|        1135131.5|    4|
|   Tecnico operativo|        1124344.0|    1|
| Recursos materiales|        1124344.0|    1|
|  Asistencial Médica|         977660.0|    1|
|Atencion y report...|         927936.0|    1|
| Procurador del agua|775717.6666666666|    3|
| procurador del agua|         735920.0|    1|
|JEFE DE MATERIA "...|        679061.69|    1|
|Distribucion y ca...|        655213.27|    2|
|Mantenimiento e i...|         649118.0|    1|
|Secretaria de dep...|644642.3428571429|    7|
|         ESTADÍSTICA|        585383.91|    1|
|JUZGADO 8o C

Average pay by department (area)

In [26]:
clean_df.groupBy('area').agg(F.avg('montobruto').alias('avg_gross')).orderBy('avg_gross', ascending=False).show(20)

+--------------------+-----------------+
|                area|        avg_gross|
+--------------------+-----------------+
|Jefatura de Gobie...|        3834794.5|
|Subsecretaría de ...|        3504155.5|
|Secretario de Pro...|2267717.285714286|
|Direccion De Obra...|       1662886.88|
|Secretaria de dep...|        1497404.0|
|  Asistencial Medica|        1315650.0|
|  Asistencial Médica|         977660.0|
|CONSEJO DEL PODER...|        675358.68|
|   Tecnica operativa|591100.6485714286|
|25FJS0010P - JEFA...|        579886.14|
|DIR GRAL ADJ DE A...|        527258.72|
|25FIS0001S - SUPE...|        527212.56|
|25DST0024A - SECU...|         520026.2|
|JUZGADO DE EJECUC...|        518591.04|
|             Virtual|        475941.89|
|LAUDOS DE  LA SEC...|         475609.6|
|25FIZ0118A - SUPE...|        471078.51|
|          08FIS0101A|        448686.95|
|JUZGADO CIVIL JIQ...|        448231.68|
|          08FIS0123M|        446905.87|
+--------------------+-----------------+
only showing top

SQL — translation practice

In [27]:
clean_df.createOrReplaceTempView('salaries_clean')
spark_job.sql("""
    SELECT entidadfederativa, COUNT(*) AS record_count, AVG(montobruto) AS avg_gross
    FROM salaries_clean
    GROUP BY entidadfederativa
    ORDER BY avg_gross DESC
""").show()

+--------------------+------------+------------------+
|   entidadfederativa|record_count|         avg_gross|
+--------------------+------------+------------------+
|             Sinaloa|        1003| 53523.21579262213|
| Baja California Sur|       23676|40842.159535394494|
|           Chihuahua|       64905|35127.899206686714|
|",LAURA CARINO SA...|           1|          34734.44|
|             Nayarit|        3225| 23797.99314418605|
| Michoacán de Ocampo|        9399|22355.339324396205|
|     Baja California|       53943|21806.607663830437|
|          Federación|      673652|19383.365629107986|
|             Tabasco|        9447| 19336.66011538053|
|              Sonora|        6841|17977.468738488533|
|",JUAN CARLOS LOP...|           1|          17213.16|
|Coahuila de Zaragoza|       41444| 16890.59274683912|
|              México|        2222|16820.915018001793|
|",SONIA OLIVAS AV...|           1|           16361.5|
|              Colima|       48628| 15843.31278748872|
|         

Report period coverage — hold off until the date columns are fixed

In [27]:
# once periodoreportainicio/periodoreportafin are correctly parsed:
clean_df.select(
    F.min('periodoreportainicio').alias('earliest_period'),
    F.max('periodoreportafin').alias('latest_period')
).show()

+---------------+-------------+
|earliest_period|latest_period|
+---------------+-------------+
|     0018-01-01|   2039-09-15|
+---------------+-------------+



Top 3 highest-paid records per state — window function

In [28]:
from pyspark.sql import functions as F

clean_df = clean_df.withColumn(
    "salary_ratio",
    F.try_divide(
        F.col("montoneto"),
        F.col("montobruto")
    )
)

In [29]:
clean_df.createOrReplaceTempView("salary_data")
spark_job.sql("""
    SELECT *,
           RANK() OVER (
               PARTITION BY entidadfederativa
               ORDER BY montobruto DESC
           ) AS salary_rank
    FROM salary_data
""").filter(
    "salary_rank <= 3"
).select(
    "entidadfederativa",
    "nombre",
    "cargo",
    "montobruto",
    "salary_rank"
).orderBy(
    "entidadfederativa",
    "salary_rank"
).show(100, truncate=False)

+----------------------------------------------------------------------------------+---------------------------------------+-----------------------------------------------------------------------------------+----------+-----------+
|entidadfederativa                                                                 |nombre                                 |cargo                                                                              |montobruto|salary_rank|
+----------------------------------------------------------------------------------+---------------------------------------+-----------------------------------------------------------------------------------+----------+-----------+
|",ANDREA JUANA LIMA ,"PROFESOR INVESTIGADOR DE ENSENANZA SUPERIOR                 | 3/4 DE TIEMPO                         |PROFESOR INVESTIGADOR DE ENSENANZA SUPERIOR, ASOCIADO "A", 3/4 DE TIEMPO, FORANEO. |14800.48  |1          |
|",CARLOS SERGIO RODRIGUEZ BENITEZ,"AUXILIAR DEL RESPONSABLE DEL CENTRO 

### FURTHER ANALYTICS

Checking for duplicates

In [30]:
clean_df = clean_df.withColumn(
    "deduction_pct",
    F.try_divide(
        F.col("deducciones_estimadas"),
        F.col("montobruto")
    ) * 100
)

In [31]:
clean_df.select(
    "montobruto",
    "montoneto",
    "deducciones_estimadas",
    "deduction_pct"
).show(10)

+----------+---------+---------------------+-------------------+
|montobruto|montoneto|deducciones_estimadas|      deduction_pct|
+----------+---------+---------------------+-------------------+
|    4254.0|   4000.0|                254.0|  5.970850963798777|
|   16092.0| 12177.86|   3914.1399999999994|  24.32351478995774|
|   16030.0|  11652.0|               4378.0|  27.31129132875858|
|   2910.65| 10180.57|             -7269.92|-249.76963908405335|
|    6188.4|  17004.4|  -10816.000000000002| -174.7786180595954|
|    2982.2|   2644.0|    338.1999999999998| 11.340621018040368|
|   4898.28|  4898.28|                  0.0|                0.0|
|     619.0|   474.96|   144.04000000000002| 23.269789983844916|
|  16812.67| 13109.02|    3703.649999999998|  22.02892223543315|
|   7579.22|   6299.3|              1279.92|   16.8872258622919|
+----------+---------+---------------------+-------------------+
only showing top 10 rows


In [36]:
total_rows = clean_df.count()

unique_rows = clean_df.dropDuplicates().count()

print("Total rows:", total_rows)
print("Duplicate rows:", total_rows - unique_rows)

Total rows: 1857546
Duplicate rows: 0


Check for invalid salary and date records

In [32]:
from pyspark.sql import functions as F

invalid_records = clean_df.filter(
    (F.col("montobruto") < 0) |
    (F.col("montoneto") < 0) |
    (F.col("period_dias") < 0) |
    (F.col("deducciones_estimadas") < 0) |
    (F.col("montoneto") > F.col("montobruto"))
)

invalid_records.show(truncate=False)

print("Invalid records:", invalid_records.count())

+--------------------+----------------------------------------------------------------------+------------------------------------+---------------------------------+---------+---------------------------------+-------------------------------------------------------------------+----------+-------------+--------------------+-----------------+-----------+---------------------+-------------------+------------------+
|entidadfederativa   |sujetoobligado                                                        |nombre                              |denominacion                     |montoneto|cargo                            |area                                                               |montobruto|idInformacion|periodoreportainicio|periodoreportafin|period_dias|deducciones_estimadas|deduction_pct      |salary_ratio      |
+--------------------+----------------------------------------------------------------------+------------------------------------+---------------------------------+--------

In [33]:
clean_df.write \
    .mode("overwrite") \
    .parquet("output/salaries_processed_parquet")

In [34]:
parquet_df = spark_job.read.parquet(
    "output/salaries_processed_parquet"
)

In [35]:
parquet_df.printSchema()

root
 |-- entidadfederativa: string (nullable = true)
 |-- sujetoobligado: string (nullable = true)
 |-- nombre: string (nullable = true)
 |-- denominacion: string (nullable = true)
 |-- montoneto: double (nullable = true)
 |-- cargo: string (nullable = true)
 |-- area: string (nullable = true)
 |-- montobruto: double (nullable = true)
 |-- idInformacion: string (nullable = true)
 |-- periodoreportainicio: date (nullable = true)
 |-- periodoreportafin: date (nullable = true)
 |-- period_dias: integer (nullable = true)
 |-- deducciones_estimadas: double (nullable = true)
 |-- deduction_pct: double (nullable = true)
 |-- salary_ratio: double (nullable = true)



In [36]:
parquet_df.show(5, truncate=False)

+-----------------+-------------------------------------------+-------------------------------+------------------------------------+---------+------------------------------------+-----------------------------------------------+----------+-------------+--------------------+-----------------+-----------+---------------------+------------------+-------------------+
|entidadfederativa|sujetoobligado                             |nombre                         |denominacion                        |montoneto|cargo                               |area                                           |montobruto|idInformacion|periodoreportainicio|periodoreportafin|period_dias|deducciones_estimadas|deduction_pct     |salary_ratio       |
+-----------------+-------------------------------------------+-------------------------------+------------------------------------+---------+------------------------------------+-----------------------------------------------+----------+-------------+------------------

In [37]:
print("Original rows:", clean_df.count())
print("Parquet rows:", parquet_df.count())

Original rows: 1857546
Parquet rows: 1857546


In [38]:
import os

# CSV file size
csv_size = os.path.getsize("salaries.csv")

# Total Parquet folder size
parquet_path = "output/salaries_processed_parquet"

parquet_size = sum(
    os.path.getsize(os.path.join(root, file))
    for root, dirs, files in os.walk(parquet_path)
    for file in files
)

print(f"CSV size: {csv_size / (1024 * 1024):.2f} MB")
print(f"Parquet size: {parquet_size / (1024 * 1024):.2f} MB")

CSV size: 391.50 MB
Parquet size: 133.16 MB


In [39]:
size_reduction = (
    (csv_size - parquet_size) / csv_size
) * 100

print(f"Size reduction: {size_reduction:.2f}%")

Size reduction: 65.99%
